In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType

# Khởi tạo Spark Session ở chế độ Local
spark = SparkSession.builder \
    .appName("Bronze_Ingestion_Chartevents_Dev") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session đã khởi tạo thành công!")

# KHAI BÁO SCHEMA CỨNG (Phải làm theo Spec để tránh OOM)
chartevents_schema = StructType([
    StructField("subject_id", IntegerType(), True),
    StructField("hadm_id", IntegerType(), True),
    StructField("stay_id", IntegerType(), True),
    StructField("caregiver_id", IntegerType(), True),
    StructField("charttime", TimestampType(), True),
    StructField("storetime", TimestampType(), True),
    StructField("itemid", IntegerType(), True),
    StructField("value", StringType(), True),
    StructField("valuenum", FloatType(), True),
    StructField("valueuom", StringType(), True),
    StructField("warning", IntegerType(), True)
])

Spark Session đã khởi tạo thành công!


In [7]:
# Đường dẫn đọc file sample đã map vào Docker
input_path = "/home/jovyan/data/raw/chartevents_sample.csv"

# Đọc CSV áp dụng schema cứng
df_raw = spark.read.csv(
    input_path, 
    header=True, 
    schema=chartevents_schema
)

print(f"Tổng số dòng đã đọc: {df_raw.count()}")
df_raw.printSchema()
df_raw.show(5)

Tổng số dòng đã đọc: 10000
root
 |-- subject_id: integer (nullable = true)
 |-- hadm_id: integer (nullable = true)
 |-- stay_id: integer (nullable = true)
 |-- caregiver_id: integer (nullable = true)
 |-- charttime: timestamp (nullable = true)
 |-- storetime: timestamp (nullable = true)
 |-- itemid: integer (nullable = true)
 |-- value: string (nullable = true)
 |-- valuenum: float (nullable = true)
 |-- valueuom: string (nullable = true)
 |-- warning: integer (nullable = true)

+----------+--------+--------+------------+-------------------+-------------------+------+-----+--------+--------+-------+
|subject_id| hadm_id| stay_id|caregiver_id|          charttime|          storetime|itemid|value|valuenum|valueuom|warning|
+----------+--------+--------+------------+-------------------+-------------------+------+-----+--------+--------+-------+
|  10000032|29079034|39553978|       47007|2180-07-23 21:01:00|2180-07-23 22:15:00|220179|   82|    82.0|    mmHg|      0|
|  10000032|29079034|395

In [10]:
# Đường dẫn ghi file Parquet
output_path = "/home/jovyan/data/bronze/mimic_iv/chartevents"

# Ghi ra Parquet với thuật toán nén Snappy
df_raw.write.parquet(
    output_path, 
    mode="overwrite", 
    compression="snappy"
)

print(f"Ingestion thành công! File đã lưu tại: {output_path}")

Ingestion thành công! File đã lưu tại: /home/jovyan/data/bronze/mimic_iv/chartevents
